# KT Movement `avg_dist` / `avg_time` 단위 검증

**문제**: KT 규격서가 두 컬럼의 단위를 모순되게 기재.

- `avg_dist`: "100m 단위" vs "m 단위" 혼재
- `avg_time`: "10분 단위" vs "분 단위" 혼재

**방법**: 행정동 polygon 중심점 간 직선거리(ground truth, EPSG:5179 meter)를 기준으로 KT `avg_dist`와 비교 → 단위 추정. `avg_time`은 추정된 거리 단위와 함께 "평균 속도가 합리적인 km/h 범위(15–30)에 들어오는 조합"으로 확정.

**증거 4개**

1. 같은 동 내 `avg_dist` 분포 (절대값 범위로 단위 추정)
2. 다른 동 `avg_dist` / 직선거리 비율 (이상적: m이면 ratio ≈ 1.2–1.5, 100m이면 ≈ 0.012–0.015)
3. 거리 구간별 ratio 일관성 (모든 구간에서 비슷해야 단위 일관)
4. 4가지 단위 조합 별 평균 속도 (15–30 km/h가 합리)

이 결과는 Phase 3-B 정제 스크립트에 단위 변환을 적용할지 결정하는 기준이 된다.

In [ ]:
import polars as pl
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data/raw')
MAPPING_DIR = Path('../data/mapping')
OUT_DIR = Path('../outputs/avgdist_unit_validation')
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi'] = 120

## 1. 데이터 로드 + 행정동 중심점 계산

Polygon은 EPSG:5179 (meter) 좌표계라서 centroid의 x/y 차이를 그대로 유클리드 거리로 쓰면 "미터" 단위 직선거리가 나온다. KT 매핑 코드는 8자리 — polygon 파일도 이미 `admdong_cd` (8자리) 컬럼을 가지고 있어 변환 불필요.

In [ ]:
df = pl.read_parquet(DATA_DIR / 'movement_sudogwon_202301.parquet')
print(f'Movement rows: {len(df):,}')

gdf = gpd.read_parquet(MAPPING_DIR / 'sudogwon_admdong_polygons.parquet')
print(f'Polygons: {len(gdf):,}  | CRS EPSG:{gdf.crs.to_epsg()}')
print(f'cols: {list(gdf.columns)}')

# centroid (m)
gdf['centroid'] = gdf.geometry.centroid
gdf['ctr_x'] = gdf.centroid.x
gdf['ctr_y'] = gdf.centroid.y

# polygon already has admdong_cd (8 digits, KT-matching).
centroids = pl.from_pandas(
    gdf[['admdong_cd', 'ctr_x', 'ctr_y']]
       .drop_duplicates(subset=['admdong_cd'])
       .reset_index(drop=True)
).with_columns(pl.col('admdong_cd').cast(pl.Utf8))
print(f'중심점 수: {len(centroids):,}')
centroids.head()

## 2. 같은 동 내부 이동 (`o == d`) 의 `avg_dist` 분포

같은 행정동 내부 이동은 보통 수백 m ~ 1–2 km 사이 (수도권 행정동 반경 약 600–1000 m). 분포의 중앙값 절대값만으로 단위가 짐작 가능:

- 100 ~ 2000 → m 단위 가능성
- 1 ~ 20 → 100m 단위 가능성
- 그 외 → 다른 단위 의심

In [ ]:
same_dong = df.filter(
    (pl.col('o_admdong_cd') == pl.col('d_admdong_cd')) &
    (pl.col('avg_dist') > 0)
)
print(f'같은 동 내 이동 행 (avg_dist>0): {len(same_dong):,} '
      f'({len(same_dong)/len(df)*100:.1f}%)')

print('\n=== 같은 동 avg_dist 분포 (행 단위, total 가중 X) ===')
print(same_dong.select([
    pl.col('avg_dist').quantile(0.05).alias('p5'),
    pl.col('avg_dist').quantile(0.25).alias('p25'),
    pl.col('avg_dist').quantile(0.50).alias('p50'),
    pl.col('avg_dist').quantile(0.75).alias('p75'),
    pl.col('avg_dist').quantile(0.95).alias('p95'),
    pl.col('avg_dist').max().alias('max'),
    pl.col('avg_dist').mean().alias('mean'),
]))

arr = same_dong['avg_dist'].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clip linear plot to 99th pct to keep things readable
p99 = float(np.quantile(arr, 0.99))
axes[0].hist(arr[arr <= p99], bins=100, color='steelblue', edgecolor='white')
axes[0].set_title(f'Same-dong avg_dist (linear, clipped at p99={p99:.1f})')
axes[0].set_xlabel('avg_dist (raw value)')
axes[0].set_ylabel('count')
axes[0].grid(True, alpha=0.3)

pos = arr[arr > 0]
axes[1].hist(np.log10(pos), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Same-dong avg_dist (log10 scale)')
axes[1].set_xlabel('log10(avg_dist)')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / '01_samedong_avgdist_dist.png', dpi=120)
plt.show()

print('\n=== 해석 가이드 ===')
print('p50 (중앙값)이:')
print('  100 ~ 2000 사이 → m 단위 가능성 (100–500m가 합리적)')
print('  1 ~ 20 사이    → 100m 단위 가능성 (1=100m, 5=500m)')
print('  그 외          → 다른 단위 의심')

## 3. 다른 동 이동 (`o != d`): `avg_dist` vs 직선거리

중심점 좌표를 조인한 뒤 직선거리(m) 계산.  비율 `ratio = avg_dist / straight_dist_m` 의 중앙값이 단위 단서:

- `ratio ≈ 1.2–1.5` → `avg_dist` 단위 = **m** (도로 우회 고려해 직선 1.2~1.5배가 정상)
- `ratio ≈ 0.012–0.015` → `avg_dist` 단위 = **100m**
- `ratio ≈ 0.001` → `avg_dist` 단위 = **km**

직선거리 < 100m인 동 쌍은 ratio 노이즈가 커서 제외.

In [ ]:
diff_dong = df.filter(
    (pl.col('o_admdong_cd') != pl.col('d_admdong_cd')) &
    (pl.col('avg_dist') > 0)
)
print(f'다른 동 이동 행 (avg_dist>0): {len(diff_dong):,}')

joined = (
    diff_dong
    .join(centroids.rename({'ctr_x': 'o_x', 'ctr_y': 'o_y'}),
          left_on='o_admdong_cd', right_on='admdong_cd', how='inner')
    .join(centroids.rename({'ctr_x': 'd_x', 'ctr_y': 'd_y'}),
          left_on='d_admdong_cd', right_on='admdong_cd', how='inner')
    .with_columns(
        ((pl.col('o_x') - pl.col('d_x')) ** 2 +
         (pl.col('o_y') - pl.col('d_y')) ** 2).sqrt().alias('straight_dist_m')
    )
    .with_columns(
        (pl.col('avg_dist') / pl.col('straight_dist_m')).alias('ratio')
    )
    .filter(pl.col('straight_dist_m') > 100)   # 인접 동 노이즈 컷
)
print(f'조인 후 행: {len(joined):,}')

print('\n=== ratio = KT avg_dist / 직선거리(m) ===')
print(joined.select([
    pl.col('ratio').quantile(0.05).alias('p5'),
    pl.col('ratio').quantile(0.25).alias('p25'),
    pl.col('ratio').quantile(0.50).alias('p50'),
    pl.col('ratio').quantile(0.75).alias('p75'),
    pl.col('ratio').quantile(0.95).alias('p95'),
    pl.col('ratio').mean().alias('mean'),
]))

print('\n=== 해석 가이드 ===')
print('  ratio ≈ 1.2–1.5 → m 단위 (도로 우회 1.2~1.5배 정상)')
print('  ratio ≈ 0.012  → 100m 단위')
print('  ratio ≈ 0.001  → km 단위')

In [ ]:
ratio_arr = joined['ratio'].to_numpy()
median_r = float(np.median(ratio_arr))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear plot: clip at a reasonable upper bound
upper = float(np.quantile(ratio_arr, 0.99))
axes[0].hist(ratio_arr[ratio_arr <= upper], bins=100,
             color='steelblue', edgecolor='white')
axes[0].axvline(median_r, color='r', linestyle='--',
                label=f'median={median_r:.4f}')
axes[0].axvline(1.0, color='g', linestyle=':', label='1.0 (m unit)')
axes[0].axvline(0.01, color='orange', linestyle=':', label='0.01 (100m unit)')
axes[0].set_title(f'ratio (linear, clipped at p99={upper:.3f})')
axes[0].set_xlabel('ratio')
axes[0].set_ylabel('count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

pos = ratio_arr[ratio_arr > 0]
axes[1].hist(np.log10(pos), bins=80, color='coral', edgecolor='white')
axes[1].axvline(np.log10(median_r), color='r', linestyle='--',
                label=f'log10(median)={np.log10(median_r):.2f}')
axes[1].axvline(0, color='g', linestyle=':', label='log10(1)=0')
axes[1].axvline(-2, color='orange', linestyle=':', label='log10(0.01)=-2')
axes[1].set_title('log10(ratio)')
axes[1].set_xlabel('log10(ratio)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / '02_ratio_distribution.png', dpi=120)
plt.show()

## 4. 거리 구간별 ratio 일관성

단위가 일관되면 직선거리가 짧든 길든 ratio가 비슷해야 한다. 구간마다 크게 달라지면 — 예: 단거리에서 0.01, 장거리에서 1 — 단위가 아니라 비선형 매핑(인코딩 문제, 절단 등)을 의심해야 함.

In [ ]:
joined_binned = joined.with_columns(
    pl.when(pl.col('straight_dist_m') < 2000).then(pl.lit('1) 0-2km'))
      .when(pl.col('straight_dist_m') < 5000).then(pl.lit('2) 2-5km'))
      .when(pl.col('straight_dist_m') < 10000).then(pl.lit('3) 5-10km'))
      .when(pl.col('straight_dist_m') < 20000).then(pl.lit('4) 10-20km'))
      .otherwise(pl.lit('5) 20km+'))
      .alias('dist_bin')
)

bin_stats = (
    joined_binned.group_by('dist_bin')
                 .agg([
                     pl.col('ratio').median().alias('ratio_p50'),
                     pl.col('ratio').quantile(0.25).alias('ratio_p25'),
                     pl.col('ratio').quantile(0.75).alias('ratio_p75'),
                     pl.col('avg_dist').median().alias('avg_dist_p50'),
                     pl.col('straight_dist_m').median().alias('straight_p50'),
                     pl.len().alias('count'),
                 ])
                 .sort('dist_bin')
)
print('=== 거리 구간별 ratio (일관성 확인) ===')
print(bin_stats)
print('\n모든 구간에서 ratio가 비슷하면 단위 일관됨. 구간마다 크게 다르면 단위/스케일 문제.')

fig, ax = plt.subplots(figsize=(10, 5))
bin_pd = bin_stats.to_pandas()
ax.bar(
    range(len(bin_pd)), bin_pd['ratio_p50'],
    yerr=[bin_pd['ratio_p50'] - bin_pd['ratio_p25'],
          bin_pd['ratio_p75'] - bin_pd['ratio_p50']],
    capsize=5, color='slateblue', alpha=0.85,
)
ax.set_xticks(range(len(bin_pd)))
ax.set_xticklabels(bin_pd['dist_bin'], rotation=15)
ax.axhline(1.0, color='g', linestyle=':', alpha=0.5, label='1.0 (m unit)')
ax.axhline(0.01, color='orange', linestyle=':', alpha=0.5, label='0.01 (100m unit)')
ax.set_ylabel('ratio median (with IQR)')
ax.set_title('Ratio consistency across distance bins')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / '03_ratio_by_bin.png', dpi=120)
plt.show()

## 5. `avg_time` 단위 검증 (속도 기반)

위에서 `avg_dist` 단위를 좁힌 뒤, `avg_time`의 단위(분 vs 10분)를 속도로 확정. 도심권 평균 속도 15–30 km/h를 합리적 범위로 본다. 4개 단위 조합(dist∈{m, 100m} × time∈{분, 10분}) 각각의 중앙 속도를 비교.

단거리(<500m)는 시간 양자화·정지·체류로 노이즈가 커서 제외.

In [ ]:
joined_speed = joined.filter(
    (pl.col('avg_time') > 0) &
    (pl.col('straight_dist_m') > 500)
)

joined_speed = joined_speed.with_columns([
    # 가정 1: dist=m, time=분
    (pl.col('avg_dist') / 1000 / (pl.col('avg_time') / 60))
        .alias('speed_kmh_dist_m_time_min'),
    # 가정 2: dist=m, time=10분
    (pl.col('avg_dist') / 1000 / (pl.col('avg_time') * 10 / 60))
        .alias('speed_kmh_dist_m_time_10min'),
    # 가정 3: dist=100m, time=분
    (pl.col('avg_dist') * 100 / 1000 / (pl.col('avg_time') / 60))
        .alias('speed_kmh_dist_100m_time_min'),
    # 가정 4: dist=100m, time=10분
    (pl.col('avg_dist') * 100 / 1000 / (pl.col('avg_time') * 10 / 60))
        .alias('speed_kmh_dist_100m_time_10min'),
])

ASSUMPTIONS = [
    ('speed_kmh_dist_m_time_min',    '가정 1: dist=m,    time=분'),
    ('speed_kmh_dist_m_time_10min',  '가정 2: dist=m,    time=10분'),
    ('speed_kmh_dist_100m_time_min', '가정 3: dist=100m, time=분'),
    ('speed_kmh_dist_100m_time_10min','가정 4: dist=100m, time=10분'),
]

print('=== 4가지 단위 가정 별 평균 속도 (km/h) ===')
print('도심 합리적 범위: 15–30 km/h\n')
stat_rows = []
for col, label in ASSUMPTIONS:
    speeds = joined_speed[col].to_numpy()
    speeds = speeds[(speeds > 0) & (speeds < 1000)]
    med = float(np.median(speeds))
    p25 = float(np.quantile(speeds, 0.25))
    p75 = float(np.quantile(speeds, 0.75))
    stat_rows.append((label, med, p25, p75))
    mark = '  ← 합리적' if 10 <= med <= 50 else ''
    print(f'{label}')
    print(f'   median: {med:8.2f} km/h   p25-p75: {p25:7.2f} - {p75:7.2f}{mark}')

fig, ax = plt.subplots(figsize=(10, 5))
labels = [s[0] for s in stat_rows]
meds = [s[1] for s in stat_rows]
p25s = [s[2] for s in stat_rows]
p75s = [s[3] for s in stat_rows]
ax.bar(range(len(labels)), meds,
       yerr=[[m - q25 for m, q25 in zip(meds, p25s)],
             [q75 - m for q75, m in zip(p75s, meds)]],
       capsize=5, color='teal', alpha=0.85)
ax.axhspan(15, 30, color='green', alpha=0.15, label='reasonable 15-30 km/h')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([l.replace('가정 ', 'Hyp ') for l in labels],
                   rotation=15, fontsize=9)
ax.set_ylabel('Median speed (km/h)')
ax.set_title('Speed under 4 unit-assumption combinations')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(OUT_DIR / '04_speed_assumptions.png', dpi=120)
plt.show()

## 6. 결론 종합

네 가지 증거를 한 번에 출력하고, 각 증거가 어떤 단위를 가리키는지 자동 판정:

1. 같은 동 `avg_dist` p50 → "m" / "100m" / "다른 단위"
2. 다른 동 ratio p50 → "m" / "100m" / "다른 단위"
3. 거리 구간별 ratio std → "일관됨" / "일관성 부족"
4. 4개 속도 조합 중 15–30 km/h 범위에 들어오는 것

1·2가 일치하면 `avg_dist` 단위 확정. 4에서 합리적 속도 조합이 정확히 하나면 `avg_time` 단위도 확정.

In [ ]:
print('=' * 60)
print('avg_dist / avg_time 단위 검증 결과 종합')
print('=' * 60)

# 1) Same-dong p50
same_dong_p50 = float(same_dong['avg_dist'].quantile(0.5))
if 100 <= same_dong_p50 <= 2000:
    verdict1 = 'm 단위 가능'
elif 1 <= same_dong_p50 <= 20:
    verdict1 = '100m 단위 가능'
else:
    verdict1 = '다른 단위 의심'
print(f'\n[1] 같은 동 avg_dist 중앙값: {same_dong_p50:.2f}  → {verdict1}')

# 2) Cross-dong ratio p50
ratio_p50 = float(joined['ratio'].quantile(0.5))
if 0.8 <= ratio_p50 <= 2.5:
    verdict2 = 'm 단위 (ratio ~ 1)'
elif 0.005 <= ratio_p50 <= 0.025:
    verdict2 = '100m 단위 (ratio ~ 0.01)'
else:
    verdict2 = '다른 단위 의심'
print(f'[2] 다른 동 ratio 중앙값      : {ratio_p50:.4f}  → {verdict2}')

# 3) Bin ratio consistency
ratio_std = float(bin_stats['ratio_p50'].std())
ratio_cv = ratio_std / float(bin_stats['ratio_p50'].mean())
verdict3 = '일관됨' if ratio_cv < 0.30 else '일관성 부족'
print(f'[3] 거리 구간별 ratio CV      : {ratio_cv:.3f} (std={ratio_std:.4f})  → {verdict3}')

# 4) Speed assumptions
print(f'\n[4] 단위 조합별 속도 (15–30 km/h 합리):')
reasonable = []
for col, label in ASSUMPTIONS:
    speeds = joined_speed[col].to_numpy()
    speeds = speeds[(speeds > 0) & (speeds < 1000)]
    med = float(np.median(speeds))
    is_ok = 10 <= med <= 50
    mark = '  ← 합리적' if is_ok else ''
    if is_ok:
        reasonable.append(label)
    print(f'    {label}: median {med:7.2f} km/h{mark}')

print('\n' + '=' * 60)
if len(reasonable) == 1:
    print(f'단일 합리적 단위 조합: {reasonable[0]}')
elif len(reasonable) == 0:
    print('합리 범위 안에 드는 조합 없음 — 도심 외 (KTX·고속도로 등) 포함 가능성')
else:
    print(f'복수 조합이 합리 범위 → 추가 정보 필요: {reasonable}')
print('=' * 60)

## 메모 — 결정 사항 (사용자가 직접 채워 넣기)

셀 실행 후 위 [1]~[4] 결과를 종합해서 아래를 직접 기록.

- `avg_dist` 단위: ☐ m  ☐ 100m  ☐ 기타 (___)
- `avg_time` 단위: ☐ 분  ☐ 10분  ☐ 기타 (___)
- 증거 일관성 (1·2·3·4가 모두 같은 단위 가리키는가):
- 의심스러운 점 (예: ratio bin별 큰 차이, 합리 조합 없음 등):
- Phase 3-B 정제 시 적용 방법:
    - 변환식 (예: `avg_dist_m = avg_dist * 100`):
    - 변환식 (예: `avg_time_min = avg_time * 10`):
- 1월 데이터로만 결론 내릴 때 한계 (예: 통근 비중 높은 1월의 편향, 설 연휴 특이값 등):